In [6]:
import requests
from bs4 import BeautifulSoup
import json


def test_single_isbn_crawling(url):
    # 차단 방지를 위한 헤더 설정
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
            '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
    }

    print(f'요청 URL: {url}')

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')

            # 방법 1: JSON-LD(구조화된 데이터)에서 ISBN 추출
            json_script = soup.find('script', type='application/ld+json')

            if json_script:
                try:
                    data = json.loads(json_script.string)

                    isbn = data.get('isbn') or data.get('productID')

                    if isbn:
                        print(f'[성공] JSON-LD에서 ISBN을 찾았습니다: {isbn}')
                        return isbn

                except Exception:
                    pass

            # 방법 2: HTML 태그에서 ISBN 추출
            isbn_element = soup.select_one(
                '.ssB_table th:-soup-contains("ISBN") + td, '
                '.book_info_inner tr td'
            )

            if isbn_element:
                isbn_text = isbn_element.get_text(strip=True)
                print(f'[성공] HTML 태그에서 ISBN을 찾았습니다: {isbn_text}')
                return isbn_text

            print(
                '[경고] 페이지는 정상적으로 불러왔으나, '
                '지정한 위치에서 ISBN을 찾지 못했습니다. '
                'HTML 구조를 확인해주세요.'
            )

        else:
            print(f'[실패] HTTP 에러 코드: {response.status_code}')

    except Exception as e:
        print(f'[에러 발생] {str(e)}')

    return None


# 테스트할 YES24 도서 상세 페이지 URL
test_url = 'https://www.yes24.com/Product/Goods/139807878'

# 크롤링 실행
test_single_isbn_crawling(test_url)

요청 URL: https://www.yes24.com/Product/Goods/139807878
[성공] JSON-LD에서 ISBN을 찾았습니다: 9791172131784


'9791172131784'

In [7]:
import re
import requests
from bs4 import BeautifulSoup


def test_book_details_crawling(url):
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
            '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
    }

    print(f'요청 URL: {url}')

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')

            # 1. 책 제목
            title_el = soup.select_one('h2.gd_name')
            title = title_el.get_text(strip=True) if title_el else '정보 없음'

            # 2. 평균 별점
            rating_el = (
                soup.select_one('span#spanGdRating.gd_rating em.yes_b')
                or soup.select_one('.gd_rating .yes_b')
            )
            rating = (
                rating_el.get_text(strip=True)
                if rating_el else '정보 없음'
            )

            # 3. 리뷰 건수
            review_el = (
                soup.select_one('span.gd_reviewCount em.txC_blue')
                or soup.select_one('.gd_reviewCount em')
            )
            review_count = (
                review_el.get_text(strip=True)
                if review_el else '정보 없음'
            )

            # 4. 판매지수 (숫자만 추출)
            sales_el = soup.select_one('span.gd_sellNum')

            if sales_el:
                sales_text = sales_el.get_text(strip=True)

                # 예: "판매지수 132,366" -> 132366
                numbers = re.findall(
                    r'\d+',
                    sales_text.replace(',', '')
                )

                sales_index = (
                    int(''.join(numbers))
                    if numbers else '정보 없음'
                )
            else:
                sales_index = '정보 없음'

            # 5. 결과 출력
            print('-' * 40)
            print(f'📚 책 제목: {title}')
            print(f'⭐ 평점: {rating}')
            print(f'💬 리뷰 건수: {review_count}')
            print(f'📈 판매지수 (숫자): {sales_index}')
            print('-' * 40)

            return {
                '상품URL': url,
                '상품명': title,
                '리뷰점수': rating,
                '리뷰건수': review_count,
                '판매지수': sales_index
            }

        else:
            print(f'[실패] HTTP 에러 코드: {response.status_code}')

    except Exception as e:
        print(f'[에러 발생] {str(e)}')

    return None


# 테스트할 YES24 도서 상세 페이지 URL
test_url = 'https://www.yes24.com/Product/Goods/139807878'

# 크롤링 실행
result = test_book_details_crawling(test_url)

요청 URL: https://www.yes24.com/Product/Goods/139807878
----------------------------------------
📚 책 제목: 빨강의 자서전
⭐ 평점: 9.6
💬 리뷰 건수: 9
📈 판매지수 (숫자): 132366
----------------------------------------


In [9]:
import pandas as pd

# 1. 엑셀 파일 경로 설정
excel_file = '../data/raw/books_raw.xlsx'

try:
    # 2. 엑셀 파일 불러오기
    df = pd.read_excel(excel_file)

    # 데이터프레임 상위 5행 출력
    print('--- [데이터프레임 상위 5개 행] ---')
    display(df.head())
    print('-' * 40)

    # 3. 컬럼 목록 확인
    print('컬럼 목록:', df.columns.tolist())

    # 4. 상품번호를 이용해 YES24 URL 생성
    if '상품번호' in df.columns:
        urls = [
            f"https://www.yes24.com/Product/Goods/{int(p_id)}"
            for p_id in df['상품번호'].dropna()
        ]

        print(f'총 {len(urls)}개의 상품(URL)이 준비되었습니다.')

        # 생성된 URL 5개 확인
        print('\n--- [생성된 URL 상위 5개] ---')
        for url in urls[:5]:
            print(url)

    else:
        print('[안내] 상품번호 컬럼이 없습니다.')
        print('실제 컬럼명을 확인해주세요.')

except FileNotFoundError:
    print(f"[에러] '{excel_file}' 파일을 찾을 수 없습니다.")

except KeyError as e:
    print(f'[에러] 해당 컬럼({e})을 찾을 수 없습니다.')

except Exception as e:
    print(f'[에러 발생] {str(e)}')

--- [데이터프레임 상위 5개 행] ---


,순위,ISBN,상품번호,상품명,판매가,YES포인트,저자,출판사,설명,출고예상일,관리분류
0,1,9791172131784,139807878,빨강의 자서전,"13,500원",750원,앤 카슨 저/민승남 역,한겨레출판,“놀라운 책이다. 몇 년간 이렇게까지 나를 흔들어놓은 글은 없었다.” _앨리스 먼로...,2,국내도서-소설/시/희곡
1,2,9791124073483,195842609,트렌드 코리아 2027,"19,800원","1,100원","김난도,전미영,최지혜,권정윤,한다혜,이혜원,전다현,이수진,서유현,이준영,이향은,김나...",미래의창,"무리에서 벗어나 변화에 앞장서는 ALPHA SHEEP\n\n‘알파alpha, α’는...",1,국내도서-경제 경영
2,3,9791141691370,197068949,가호,"16,650원",920원,무라카미 하루키 저/홍은주 역,문학동네,무라카미 하루키 3년 만의 최신 장편!\n\n‘좋은 이야기란 무엇인가’\n무라카미 ...,6,국내도서-소설/시/희곡
3,4,9791198987679,197772287,가짜 돈 진짜 부,"22,500원","1,250원",박종훈 저,글로퍼스,"“주가가 오른 게 아니라 돈이 폭락한 것이다.”\n모아도 가난해지고, 빌려도 위험해...",2,국내도서-경제 경영
4,5,9791169666886,196557643,2027 공단기 소방단기 써니 행정법총론 기출문제집,"41,400원",460원,박준철 편저,공단기(에스티유니타스),“더 효율적인 학습을 위해 중복지문은 덜어내고 꼭 필요한 기출지문을 알차게 담았습니...,1,국내도서-수험서 자격증


----------------------------------------
컬럼 목록: ['순위', 'ISBN', '상품번호', '상품명', '판매가', 'YES포인트', '저자', '출판사', '설명', '출고예상일', '관리분류']
총 206개의 상품(URL)이 준비되었습니다.

--- [생성된 URL 상위 5개] ---
https://www.yes24.com/Product/Goods/139807878
https://www.yes24.com/Product/Goods/195842609
https://www.yes24.com/Product/Goods/197068949
https://www.yes24.com/Product/Goods/197772287
https://www.yes24.com/Product/Goods/196557643


In [12]:
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# 1. 파일 경로 설정
excel_file = Path('../data/raw/books_raw.xlsx')
ext_file = Path('../data/raw/books_raw_ext.csv')

# 2. 원본 데이터 불러오기
df = pd.read_excel(excel_file)

print(f'원본 데이터: {len(df)}권')
print('컬럼:', df.columns.tolist())

# 3. URL 생성
# 주의: 상품번호가 YES24 상품 ID인지 먼저 확인해야 합니다.
df['url'] = df['상품번호'].apply(
    lambda x: (
        f'https://www.yes24.com/Product/Goods/{int(x)}'
        if pd.notna(x) else None
    )
)

# 4. 수집 함수
def crawl_book(url):
    book_info = {
        'url': url,
        '평점': None,
        '리뷰건수': None,
        '판매지수': None,
        '수집상태': '미수집'
    }

    if not url:
        return book_info

    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        )
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        if response.status_code != 200:
            book_info['수집상태'] = f'HTTP {response.status_code}'
            return book_info

        soup = BeautifulSoup(response.text, 'html.parser')

        # ① 평점
        rating_el = (
            soup.select_one(
                'span#spanGdRating.gd_rating em.yes_b'
            )
            or soup.select_one('.gd_rating .yes_b')
        )

        if rating_el:
            book_info['평점'] = rating_el.get_text(strip=True)

        # ② 리뷰 건수
        review_el = (
            soup.select_one(
                'span.gd_reviewCount em.txC_blue'
            )
            or soup.select_one('.gd_reviewCount em')
        )

        if review_el:
            book_info['리뷰건수'] = review_el.get_text(strip=True)

        # ③ 판매지수
        sales_el = soup.select_one('span.gd_sellNum')

        if sales_el:
            sales_text = sales_el.get_text(strip=True)
            match = re.search(r'[\d,]+', sales_text)

            if match:
                book_info['판매지수'] = int(
                    match.group().replace(',', '')
                )

        book_info['수집상태'] = 'HTTP 200'

    except requests.RequestException as e:
        book_info['수집상태'] = f'요청 오류: {e}'

    except Exception as e:
        book_info['수집상태'] = f'파싱 오류: {e}'

    return book_info


# 5. 전체 도서 크롤링
# 처음에는 3권만 테스트한 후 전체로 변경하세요.
urls = df['url'].dropna().tolist()

extra_results = []

for url in tqdm(urls, desc='도서 정보 수집'):
    result = crawl_book(url)
    extra_results.append(result)

    time.sleep(1)

# 6. 외부 데이터 저장
extra_df = pd.DataFrame(extra_results)

ext_file.parent.mkdir(parents=True, exist_ok=True)

extra_df.to_csv(
    ext_file,
    index=False,
    encoding='utf-8-sig'
)

print('\n외부 데이터 저장 완료:', ext_file)

display(extra_df.head())

원본 데이터: 206권
컬럼: ['순위', 'ISBN', '상품번호', '상품명', '판매가', 'YES포인트', '저자', '출판사', '설명', '출고예상일', '관리분류']


도서 정보 수집:   0%|          | 0/206 [00:00<?, ?it/s]


외부 데이터 저장 완료: ..\data\raw\books_raw_ext.csv


,url,평점,리뷰건수,판매지수,수집상태
0,https://www.yes24.com/Product/Goods/139807878,9.6,9,132366.0,HTTP 200
1,https://www.yes24.com/Product/Goods/195842609,9.4,77,249162.0,HTTP 200
2,https://www.yes24.com/Product/Goods/197068949,10.0,10,235206.0,HTTP 200
3,https://www.yes24.com/Product/Goods/197772287,10.0,1,80070.0,HTTP 200
4,https://www.yes24.com/Product/Goods/196557643,9.7,43,240237.0,HTTP 200


In [13]:
import pandas as pd
from pathlib import Path

# 1. 파일 경로
raw_path = Path('../data/raw/books_raw.xlsx')
ext_path = Path('../data/raw/books_raw_ext.csv')
output_path = Path('../data/processed/books_best_total.csv')

# 2. 데이터 불러오기
df_raw = pd.read_excel(raw_path)
df_ext = pd.read_csv(ext_path)

# 3. 원본에 URL 생성
df_raw['url'] = df_raw['상품번호'].apply(
    lambda x: (
        f'https://www.yes24.com/Product/Goods/{int(x)}'
        if pd.notna(x) else None
    )
)

# 4. URL 중복 확인
print('외부 데이터 URL 중복:', df_ext['url'].duplicated().sum())

# 5. 병합
merged_df = pd.merge(
    df_raw,
    df_ext,
    on='url',
    how='left',
    validate='many_to_one',
    indicator=True
)

# 6. 병합 검증
print('원본 데이터:', len(df_raw))
print('외부 데이터:', len(df_ext))
print('병합 데이터:', len(merged_df))

print('\n병합 상태:')
print(merged_df['_merge'].value_counts())

assert len(merged_df) == len(df_raw), '병합 후 행 수가 달라졌습니다.'

# 7. 병합 상태 컬럼 제거
merged_df = merged_df.drop(columns=['_merge'])

# 8. 저장
output_path.parent.mkdir(parents=True, exist_ok=True)

merged_df.to_csv(
    output_path,
    index=False,
    encoding='utf-8-sig'
)

print('\n병합 파일 저장 완료!')
print('저장 경로:', output_path)

# 9. 결과 확인
display(
    merged_df[
        ['상품명', '평점', '리뷰건수', '판매지수']
    ].head(10)
)

외부 데이터 URL 중복: 0
원본 데이터: 206
외부 데이터: 206
병합 데이터: 206

병합 상태:
_merge
both          206
left_only       0
right_only      0
Name: count, dtype: int64

병합 파일 저장 완료!
저장 경로: ..\data\processed\books_best_total.csv


,상품명,평점,리뷰건수,판매지수
0,빨강의 자서전,9.6,9,132366.0
1,트렌드 코리아 2027,9.4,77,249162.0
2,가호,10.0,10,235206.0
3,가짜 돈 진짜 부,10.0,1,80070.0
4,2027 공단기 소방단기 써니 행정법총론 기출문제집,9.7,43,240237.0
5,쇼펜하우어 인생수업 (30만 부 기념 개정증보판),8.9,280,300552.0
6,"세네카, 오늘을 빼앗기고 있는 당신에게",8.2,255,490788.0
7,싯다르타,9.4,"1,035",445071.0
8,남편의 아름다움,8.0,4,56478.0
9,전지적 독자 시점 문고본 세트,10.0,5,51660.0


In [14]:
print('전체 데이터 크기:', merged_df.shape)

print('\n추가 수집 컬럼 결측치:')
print(
    merged_df[
        ['평점', '리뷰건수', '판매지수']
    ].isna().sum()
)

print('\n병합 결과 미리보기:')
display(merged_df.head())

전체 데이터 크기: (206, 16)

추가 수집 컬럼 결측치:
평점      13
리뷰건수    13
판매지수     2
dtype: int64

병합 결과 미리보기:


,순위,ISBN,상품번호,상품명,판매가,YES포인트,저자,출판사,설명,출고예상일,관리분류,url,평점,리뷰건수,판매지수,수집상태
0,1,9791172131784,139807878,빨강의 자서전,"13,500원",750원,앤 카슨 저/민승남 역,한겨레출판,“놀라운 책이다. 몇 년간 이렇게까지 나를 흔들어놓은 글은 없었다.” _앨리스 먼로...,2,국내도서-소설/시/희곡,https://www.yes24.com/Product/Goods/139807878,9.6,9,132366.0,HTTP 200
1,2,9791124073483,195842609,트렌드 코리아 2027,"19,800원","1,100원","김난도,전미영,최지혜,권정윤,한다혜,이혜원,전다현,이수진,서유현,이준영,이향은,김나...",미래의창,"무리에서 벗어나 변화에 앞장서는 ALPHA SHEEP\n\n‘알파alpha, α’는...",1,국내도서-경제 경영,https://www.yes24.com/Product/Goods/195842609,9.4,77,249162.0,HTTP 200
2,3,9791141691370,197068949,가호,"16,650원",920원,무라카미 하루키 저/홍은주 역,문학동네,무라카미 하루키 3년 만의 최신 장편!\n\n‘좋은 이야기란 무엇인가’\n무라카미 ...,6,국내도서-소설/시/희곡,https://www.yes24.com/Product/Goods/197068949,10.0,10,235206.0,HTTP 200
3,4,9791198987679,197772287,가짜 돈 진짜 부,"22,500원","1,250원",박종훈 저,글로퍼스,"“주가가 오른 게 아니라 돈이 폭락한 것이다.”\n모아도 가난해지고, 빌려도 위험해...",2,국내도서-경제 경영,https://www.yes24.com/Product/Goods/197772287,10.0,1,80070.0,HTTP 200
4,5,9791169666886,196557643,2027 공단기 소방단기 써니 행정법총론 기출문제집,"41,400원",460원,박준철 편저,공단기(에스티유니타스),“더 효율적인 학습을 위해 중복지문은 덜어내고 꼭 필요한 기출지문을 알차게 담았습니...,1,국내도서-수험서 자격증,https://www.yes24.com/Product/Goods/196557643,9.7,43,240237.0,HTTP 200
